#**ML Challenge - IEEE SB GEHU | Fault Detection System**


## **Problem Statement -**
We are given sensor data from an embedded device monitoring system.
The goal is to predict whether a device is operating normally or has a fault.

- **Input:** 47 numerical features (F01–F47) representing operational parameters
- **Target:** Binary Classification
  - Class 0 → Normal device
  - Class 1 → Faulty device

## Evaluation Metrics
- Accuracy
- F1-Score

## Dataset
- Training set: ~43,776 rows
- Test set: ~11,000 rows
- No missing values
- Clean numerical data


In [ ]:
# Core libraries
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# ML - Preprocessing
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# ML - Models
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
import xgboost as xgb
import lightgbm as lgb
!pip install optuna

# Utility
import warnings
warnings.filterwarnings('ignore')

print("All libraries loaded successfully!")

In [ ]:
from google.colab import files
uploaded = files.upload()

# Load datasets
train_df = pd.read_csv('TRAIN.csv')
test_df = pd.read_csv('TEST.csv')

print(f"Train shape: {train_df.shape}")
print(f"Test shape:  {test_df.shape}")

In [ ]:
print("let's Explore Training Data")
print(train_df.head())

print(f"Total Rows     : {train_df.shape[0]}")
print(f"Total Features : {train_df.shape[1] - 1}")
print(f"Target Column  : Class")



In [ ]:

print("Missing Values:", train_df.isnull().sum().sum())
print("Duplicate Rows:", train_df.duplicated().sum())

#Data Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
train_df['Class'].value_counts().plot(kind='bar', ax=axes[0],
                                       color=['steelblue','tomato'],
                                       edgecolor='black')
axes[0].set_title('Class Distribution (Count)')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(['Normal (0)', 'Faulty (1)'], rotation=0)

# Pie chart
train_df['Class'].value_counts().plot(kind='pie', ax=axes[1],
                                       labels=['Normal (0)', 'Faulty (1)'],
                                       colors=['steelblue','tomato'],
                                       autopct='%1.1f%%', startangle=90)
axes[1].set_title('Class Distribution (%)')
axes[1].set_ylabel('')

plt.suptitle('Target Variable Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
print(f"Total rows in dataset: {train_df.shape[0]}")
print(f"Duplicate rows found : {train_df.duplicated().sum()}")

# Show some duplicate examples
print("Sample duplicate rows:")
display(train_df[train_df.duplicated(keep=False)].head(6))

In [ ]:
# Some Examples of duplicate rows
dupes = train_df[train_df.duplicated(keep=False)]
dupes_sorted = dupes.sort_values(by=list(dupes.columns))
print(f"Total flagged duplicate rows: {len(dupes_sorted)}")
display(dupes_sorted.head(6))


In [ ]:
# Calculate variance of every feature
variances = train_df.drop('Class', axis=1).var().sort_values()

print("Bottom 10 features by variance (lowest = least useful):")
print(variances.head(10))

# Visualize
plt.figure(figsize=(10, 4))
variances.head(20).plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Bottom 20 Features by Variance')
plt.ylabel('Variance')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
corr_matrix = train_df.drop('Class', axis=1).corr().abs()

# Find highly correlated pairs
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_pairs = [(col, row, upper.loc[row, col])
                   for col in upper.columns
                   for row in upper.index
                   if upper.loc[row, col] > 0.95]

print(f"Highly correlated pairs (correlation > 0.95): {len(high_corr_pairs)}")
print(f"\n{'Feature A':<10} {'Feature B':<10} {'Correlation'}")
print("-" * 35)
for a, b, corr in high_corr_pairs:
    print(f"{a:<10} {b:<10} {corr:.4f}")

In [ ]:
# Focus heatmap on the suspicious pairs we found
focus_cols = ['F04','F05','F06','F07','F08','F09',
              'F24','F25','F26','F27','F28','F29']

plt.figure(figsize=(10, 8))
sns.heatmap(
    train_df[focus_cols].corr(),
    annot=True, fmt='.2f',
    cmap='coolwarm', center=0,
    square=True, linewidths=0.5
)
plt.title('Correlation Heatmap — Suspected Redundant Features', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Features with extreme outliers
stats = train_df.drop('Class', axis=1).describe().T
stats['max_to_mean_ratio'] = stats['max'] / (stats['mean'].abs() + 1e-9)
outlier_features = stats[stats['max_to_mean_ratio'] > 100].sort_values('max_to_mean_ratio', ascending=False)

print("Features with extreme outliers (max >> mean):")
display(outlier_features[['mean', 'max', 'std', 'max_to_mean_ratio']])

# Boxplot for top 6 most extreme
top_outlier_cols = outlier_features.head(6).index.tolist()
plt.figure(figsize=(14, 5))
for i, col in enumerate(top_outlier_cols):
    plt.subplot(1, 6, i+1)
    plt.boxplot(train_df[col].dropna(), patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.7))
    plt.title(col, fontsize=10)
    plt.xticks([])
plt.suptitle('Boxplots — Extreme Outlier Features', fontsize=13)
plt.tight_layout()
plt.show()

#Data Cleaning

In [ ]:
print(f"Shape before: {train_df.shape}")
train_df = train_df.drop_duplicates()
print(f"Shape after : {train_df.shape}")
print(f"Removed {738} duplicate rows")

In [ ]:
# F20  → near zero variance (carries no information)
# F24-F29 → highly correlated with F04-F09 (redundant)

cols_to_drop = ['F20', 'F24', 'F25', 'F26', 'F27', 'F28', 'F29']

train_df = train_df.drop(columns=cols_to_drop, errors='ignore')
test_df  = test_df.drop(columns=cols_to_drop, errors='ignore')

print(f"Dropped: {cols_to_drop}")
print(f"Remaining features: {train_df.shape[1] - 1}")

In [ ]:
X = train_df.drop(columns=['Class'])
y = train_df['Class']

# Keep ID for final submission
test_ids = test_df['ID']
X_test   = test_df.drop(columns=['ID'])

print(f" X shape      : {X.shape}")
print(f" y shape      : {y.shape}")
print(f" X_test shape : {X_test.shape}")

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,     # 80% training, 20% validation
    random_state=42,   # reproducibility
    stratify=y         # maintain 60/40 ratio in both splits
)

print(f"X_train : {X_train.shape}")
print(f"X_val   : {X_val.shape}")
print(f"\nClass ratio in training:")
print(y_train.value_counts(normalize=True).mul(100).round(1))
print(f"\nClass ratio in validation:")
print(y_val.value_counts(normalize=True).mul(100).round(1))

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
import time

# Dictionary of models to compare
models = {
    'Logistic Regression' : LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1),
    'Decision Tree'       : DecisionTreeClassifier(random_state=42),
    'Random Forest'       : RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'SVM'                 : SVC(random_state=42),
    'XGBoost'             : XGBClassifier(n_estimators=300, random_state=42,
                                          n_jobs=-1, verbosity=0, eval_metric='logloss'),
    'LightGBM'            : LGBMClassifier(n_estimators=300, random_state=42,
                                           n_jobs=-1, verbosity=-1)
}

results = []

print("Training and evaluating all models...")
print("This will take a few minutes (SVM is slow)...\n")

for name, model in models.items():
    start = time.time()

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_val)

    # Score
    acc  = accuracy_score(y_val, y_pred) * 100
    f1   = f1_score(y_val, y_pred) * 100
    elapsed = time.time() - start

    results.append({
        'Model'        : name,
        'Accuracy'     : acc,
        'F1 Score'     : f1,
        'Training Time': f"{elapsed:.1f}s"
    })

    print(f"{name:<25} Accuracy: {acc:.2f}%  F1: {f1:.2f}%  Time: {elapsed:.1f}s")

print("\nAll models evaluated!")

In [ ]:
results_df = pd.DataFrame(results).sort_values('F1 Score', ascending=False)

print("\n" + "=" * 65)
print("            Model Selection Comparison")
print("=" * 65)
print(f"{'Model':<25} {'Accuracy':>10} {'F1 Score':>10} {'Time':>10}")
print("-" * 65)
for _, row in results_df.iterrows():
    print(f"{row['Model']:<25} {row['Accuracy']:>9.2f}% {row['F1 Score']:>9.2f}% {row['Training Time']:>10}")
print("=" * 65)

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# F1 Score comparison
colors = ['gold' if i == 0 else 'steelblue' for i in range(len(results_df))]
axes[0].barh(results_df['Model'], results_df['F1 Score'],
             color=colors, edgecolor='black')
axes[0].set_xlabel('F1 Score (%)')
axes[0].set_title('Model Comparison — F1 Score')
axes[0].set_xlim(results_df['F1 Score'].min() - 5, 101)
for i, (val, name) in enumerate(zip(results_df['F1 Score'], results_df['Model'])):
    axes[0].text(val + 0.1, i, f'{val:.2f}%', va='center', fontsize=9)

# Accuracy comparison
axes[1].barh(results_df['Model'], results_df['Accuracy'],
             color=colors, edgecolor='black')
axes[1].set_xlabel('Accuracy (%)')
axes[1].set_title('Model Comparison — Accuracy')
axes[1].set_xlim(results_df['Accuracy'].min() - 5, 101)
for i, (val, name) in enumerate(zip(results_df['Accuracy'], results_df['Model'])):
    axes[1].text(val + 0.1, i, f'{val:.2f}%', va='center', fontsize=9)

plt.suptitle('Why XGBoost & LightGBM? — Model Selection Analysis',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Print winner
best_model = results_df.iloc[0]
print(f"\nBest Model: {best_model['Model']}")
print(f"   F1 Score : {best_model['F1 Score']:.2f}%")
print(f"   Accuracy : {best_model['Accuracy']:.2f}%")
print(f"\nThis is why we chose XGBoost & LightGBM!")


#Training Time :-

In [ ]:
# Initialize model
xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=26465/17311,  # handle class imbalance
    random_state=42,
    n_jobs=-1,                     # use all CPU cores
    eval_metric='logloss',
    verbosity=0
)

# Train
print("Training XGBoost...")
xgb_model.fit(X_train, y_train)
print("Training complete!")

In [ ]:
# Predictions on validation set
y_pred_xgb = xgb_model.predict(X_val)

# Scores
acc_xgb = accuracy_score(y_val, y_pred_xgb)
f1_xgb  = f1_score(y_val, y_pred_xgb)

print("=" * 40)
print("       XGBoost Baseline Results")
print("=" * 40)
print(f"  Accuracy : {acc_xgb*100:.2f}%")
print(f"  F1 Score : {f1_xgb*100:.2f}%")
print("=" * 40)

print("\nDetailed Classification Report:")
print(classification_report(y_val, y_pred_xgb,
      target_names=['Normal (0)', 'Faulty (1)']))

In [ ]:
cm = confusion_matrix(y_val, y_pred_xgb)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal (0)', 'Faulty (1)'],
            yticklabels=['Normal (0)', 'Faulty (1)'])
plt.title('XGBoost — Confusion Matrix', fontsize=13)
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

# Explain the confusion matrix
tn, fp, fn, tp = cm.ravel()
print(f"\nTrue Normal  (Correct Normal predictions) : {tn}")
print(f" True Faulty  (Correct Faulty predictions) : {tp}")
print(f" False Alarm  (Normal predicted as Faulty) : {fp}")
print(f" Missed Fault (Faulty predicted as Normal) : {fn}")

In [ ]:
# Get feature importances
importance_df = pd.DataFrame({
    'Feature'   : X_train.columns,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("Top 15 Most Important Features:")
display(importance_df.head(15))

# Plot
plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df.head(15),
            x='Importance', y='Feature',
            palette='viridis')
plt.title('XGBoost — Top 15 Feature Importances', fontsize=13)
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

#Let's Fine Tune it

In [ ]:
# Initialize model
lgb_model = LGBMClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=26465/17311,  # handle class imbalance
    random_state=42,
    n_jobs=-1,
    verbosity=-1                   # suppress warnings
)

# Train
print("Training LightGBM...")
lgb_model.fit(X_train, y_train)
print("Training complete!")

In [ ]:
# Predictions
y_pred_lgb = lgb_model.predict(X_val)

# Scores
acc_lgb = accuracy_score(y_val, y_pred_lgb)
f1_lgb  = f1_score(y_val, y_pred_lgb)

print("=" * 40)
print("       LightGBM Baseline Results")
print("=" * 40)
print(f"  Accuracy : {acc_lgb*100:.2f}%")
print(f"  F1 Score : {f1_lgb*100:.2f}%")
print("=" * 40)

print("\nDetailed Classification Report:")
print(classification_report(y_val, y_pred_lgb,
      target_names=['Normal (0)', 'Faulty (1)']))


In [ ]:
print("=" * 45)
print("         Model Comparison Summary")
print("=" * 45)
print(f"{'Model':<15} {'Accuracy':>10} {'F1 Score':>10}")
print("-" * 45)
print(f"{'XGBoost':<15} {acc_xgb*100:>9.2f}% {f1_xgb*100:>9.2f}%")
print(f"{'LightGBM':<15} {acc_lgb*100:>9.2f}% {f1_lgb*100:>9.2f}%")
print("=" * 45)

# Visual comparison
models   = ['XGBoost', 'LightGBM']
accuracy = [acc_xgb*100, acc_lgb*100]
f1scores = [f1_xgb*100, f1_lgb*100]

x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - width/2, accuracy, width,
               label='Accuracy', color='steelblue', edgecolor='black')
bars2 = ax.bar(x + width/2, f1scores, width,
               label='F1 Score', color='tomato', edgecolor='black')

# Add value labels on bars
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() - 0.3,
            f'{bar.get_height():.2f}%',
            ha='center', va='top',
            fontsize=10, color='white', fontweight='bold')

for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() - 0.3,
            f'{bar.get_height():.2f}%',
            ha='center', va='top',
            fontsize=10, color='white', fontweight='bold')

ax.set_ylim(95, 100)
ax.set_ylabel('Score (%)')
ax.set_title('XGBoost vs LightGBM — Baseline Comparison', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=12)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
import optuna
from sklearn.model_selection import StratifiedKFold, cross_val_score
from xgboost import XGBClassifier # Import XGBClassifier as it's used in objective_xgb
from sklearn.metrics import f1_score # Import f1_score as it's used in objective_xgb

def objective_xgb(trial):
    params = {
        'n_estimators'     : trial.suggest_int('n_estimators', 100, 500),
        'max_depth'        : trial.suggest_int('max_depth', 3, 10),
        'learning_rate'    : trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample'        : trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight' : trial.suggest_int('min_child_weight', 1, 10),
        'scale_pos_weight' : 26465/17311,
        'random_state'     : 42,
        'n_jobs'           : -1,
        'verbosity'        : 0,
        'eval_metric'      : 'logloss'
    }

    model = XGBClassifier(**params)

    # Single split instead of 5-fold (3x faster)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    return f1_score(y_val, y_pred)

sampler = optuna.samplers.TPESampler(seed=42)
study_xgb = optuna.create_study(direction='maximize', sampler=sampler)
study_xgb.optimize(objective_xgb, n_trials=30, show_progress_bar=True)

In [ ]:
def objective_lgb(trial):
    params = {
        'n_estimators'     : trial.suggest_int('n_estimators', 100, 500),
        'max_depth'        : trial.suggest_int('max_depth', 3, 10),
        'learning_rate'    : trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample'        : trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'num_leaves'       : trial.suggest_int('num_leaves', 20, 150),
        'reg_alpha'        : trial.suggest_float('reg_alpha', 0, 5),
        'reg_lambda'       : trial.suggest_float('reg_lambda', 0, 5),
        'scale_pos_weight' : 26465/17311,
        'random_state'     : 42,
        'n_jobs'           : -1,
        'verbosity'        : -1
    }

    model = LGBMClassifier(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    return f1_score(y_val, y_pred)

print("Tuning LightGBM (30 trials)...")
sampler   = optuna.samplers.TPESampler(seed=42)
study_lgb = optuna.create_study(direction='maximize', sampler=sampler)
study_lgb.optimize(objective_lgb, n_trials=30, show_progress_bar=True)

print(f"\nLightGBM Tuning Complete!")
print(f"   Best F1 Score : {study_lgb.best_value*100:.2f}%")
print(f"   Best Params   : {study_lgb.best_params}")

In [ ]:
# XGBoost with best params
print("Training tuned XGBoost...")
best_xgb = XGBClassifier(
    **study_xgb.best_params,
    scale_pos_weight=26465/17311,
    random_state=42,
    n_jobs=-1,
    verbosity=0,
    eval_metric='logloss'
)
best_xgb.fit(X_train, y_train)
print("Tuned XGBoost ready!")

# LightGBM with best params
print("\nTraining tuned LightGBM...")
best_lgb = LGBMClassifier(
    **study_lgb.best_params,
    scale_pos_weight=26465/17311,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)
best_lgb.fit(X_train, y_train)
print("Tuned LightGBM ready!")

In [ ]:
# Tuned predictions
y_pred_xgb_tuned = best_xgb.predict(X_val)
y_pred_lgb_tuned = best_lgb.predict(X_val)

acc_xgb_tuned = accuracy_score(y_val, y_pred_xgb_tuned)
f1_xgb_tuned  = f1_score(y_val, y_pred_xgb_tuned)
acc_lgb_tuned = accuracy_score(y_val, y_pred_lgb_tuned)
f1_lgb_tuned  = f1_score(y_val, y_pred_lgb_tuned)

print("=" * 55)
print("           Full Model Comparison")
print("=" * 55)
print(f"{'Model':<25} {'Accuracy':>10} {'F1 Score':>10}")
print("-" * 55)
print(f"{'XGBoost (baseline)':<25} {acc_xgb*100:>9.2f}% {f1_xgb*100:>9.2f}%")
print(f"{'LightGBM (baseline)':<25} {acc_lgb*100:>9.2f}% {f1_lgb*100:>9.2f}%")
print("-" * 55)
print(f"{'XGBoost (tuned)':<25} {acc_xgb_tuned*100:>9.2f}% {f1_xgb_tuned*100:>9.2f}%")
print(f"{'LightGBM (tuned)':<25} {acc_lgb_tuned*100:>9.2f}% {f1_lgb_tuned*100:>9.2f}%")
print("=" * 55)

In [ ]:
# Get probabilities from both tuned models
prob_xgb = best_xgb.predict_proba(X_val)[:, 1]
prob_lgb  = best_lgb.predict_proba(X_val)[:, 1]

# Average probabilities (soft voting)
avg_probs       = (prob_xgb + prob_lgb) / 2
y_pred_ensemble = (avg_probs >= 0.5).astype(int)

# Scores
acc_ensemble = accuracy_score(y_val, y_pred_ensemble)
f1_ensemble  = f1_score(y_val, y_pred_ensemble)

print("=" * 55)
print("           FINAL Model Comparison")
print("=" * 55)
print(f"{'Model':<25} {'Accuracy':>10} {'F1 Score':>10}")
print("-" * 55)
print(f"{'XGBoost (baseline)':<25} {acc_xgb*100:>9.2f}% {f1_xgb*100:>9.2f}%")
print(f"{'LightGBM (baseline)':<25} {acc_lgb*100:>9.2f}% {f1_lgb*100:>9.2f}%")
print("-" * 55)
print(f"{'XGBoost (tuned)':<25} {acc_xgb_tuned*100:>9.2f}% {f1_xgb_tuned*100:>9.2f}%")
print(f"{'LightGBM (tuned)':<25} {acc_lgb_tuned*100:>9.2f}% {f1_lgb_tuned*100:>9.2f}%")
print("-" * 55)
print(f"{'ENSEMBLE (XGB+LGB)':<25} {acc_ensemble*100:>9.2f}% {f1_ensemble*100:>9.2f}%")
print("=" * 55)

In [ ]:
# Give more weight to better model (XGBoost)
weights = [0.6, 0.4]  # 60% XGBoost, 40% LightGBM

weighted_probs       = (weights[0] * prob_xgb) + (weights[1] * prob_lgb)
y_pred_weighted      = (weighted_probs >= 0.5).astype(int)

acc_weighted = accuracy_score(y_val, y_pred_weighted)
f1_weighted  = f1_score(y_val, y_pred_weighted)

print(f"Weighted Ensemble (60/40) → Accuracy: {acc_weighted*100:.2f}% | F1: {f1_weighted*100:.2f}%")
print(f"Equal Ensemble    (50/50) → Accuracy: {acc_ensemble*100:.2f}% | F1: {f1_ensemble*100:.2f}%")
print(f"XGBoost alone             → Accuracy: {acc_xgb_tuned*100:.2f}% | F1: {f1_xgb_tuned*100:.2f}%")

# Pick best
scores = {
    'XGBoost alone'   : f1_xgb_tuned,
    'Equal Ensemble'  : f1_ensemble,
    'Weighted Ensemble': f1_weighted
}
best_approach = max(scores, key=scores.get)
print(f"\n Best approach: {best_approach} with F1: {scores[best_approach]*100:.2f}%")


In [ ]:
# Use best approach for confusion matrix
cm = confusion_matrix(y_val, y_pred_weighted)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Normal (0)', 'Faulty (1)'],
            yticklabels=['Normal (0)', 'Faulty (1)'])
plt.title('Ensemble Model — Final Confusion Matrix', fontsize=13)
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"\nTrue Normal   : {tn}")
print(f"True Faulty   : {tp}")
print(f"False Alarm   : {fp}")
print(f"Missed Faults : {fn}")
print(f"\nImprovement from baseline:")
print(f"  Missed Faults : 90 → {fn} ({90-fn} fewer missed faults!)")
print(f"  False Alarms  : 59 → {fp} ({59-fp} fewer false alarms!)")

In [ ]:
# Get probabilities from both tuned models for the test set
prob_xgb_test = best_xgb.predict_proba(X_test)[:, 1]
prob_lgb_test  = best_lgb.predict_proba(X_test)[:, 1]

# Average probabilities (soft voting) for the test set
avg_probs_test       = (prob_xgb_test + prob_lgb_test) / 2
y_pred_ensemble_test = (avg_probs_test >= 0.5).astype(int)

# Create the submission DataFrame
submission = pd.DataFrame({
    'ID': test_ids,
    'Class': y_pred_ensemble_test
})

# Check TEST.csv structure
print("TEST.csv structure:")
print(f"   Rows    : {len(test_df)}")
print(f"   Columns : {list(test_df.columns)}")
display(test_df.head(5))

# Check submission matches exactly
print("\nVerifying FINAL.csv format:")
print(f"   Total rows in TEST.csv   : {len(test_df)}")
print(f"   Total rows in FINAL.csv  : {len(submission)}")
print(f"   Row count match          : {len(test_df) == len(submission)}")

# Check ID order matches exactly
print(f"\n   First 5 IDs in TEST.csv  : {test_df['ID'].head().tolist()}")
print(f"   First 5 IDs in FINAL.csv : {submission['ID'].head().tolist()}")
print(f"   ID order match           : {test_df['ID'].tolist() == submission['ID'].tolist()}")

# Check column names match expected format
print(f"\n   Columns in FINAL.csv     : {list(submission.columns)}")
print(f"   Expected                 : ['ID', 'Class']")

# Preview final submission
print("\nFinal submission preview:")
display(submission.head(10))


In [ ]:
submission.to_csv('FINAL.csv', index=False)
print("\nFINAL.csv saved!")

from google.colab import files
files.download('FINAL.csv')
print("Download started! Check your downloads folder!")